Left to do:
1. Import random stuff that we no longer have, like torch OK
2. Build the train and validation datasets OK
3. Create eval function to save best model and get eval loss OK
4. Add evals like BLEU, METEOR, ROUGE, or CIDEr(optional if I read it right)
5. Train it

In [1]:
import kagglehub
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import torchvision.transforms as transfroms
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
from PIL import Image
import pandas as pd
import os
import json
from sklearn.model_selection import train_test_split
from collections import Counter
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from torch.nn.utils.rnn import pad_sequence


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1778490217.502174  115586 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778490217.576603  115586 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778490219.840193  115586 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point roun

In [2]:
# Create the dataset

In [3]:
path = kagglehub.dataset_download("nikhil7280/coco-image-caption")
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/nikhil7280/coco-image-caption/versions/1


In [4]:
print(os.listdir(path))


['val2017', 'annotations_trainval2014', 'annotations_trainval2017', 'train2014']


In [7]:
train_img_dir = os.path.join(path, "train2014", "train2014")
ann_path = os.path.join(
    path,
    "annotations_trainval2014",
    "annotations",
    "captions_train2014.json"
)

In [8]:
with open(ann_path, "r") as f:
    data = json.load(f)

In [9]:
image_id_to_file = {
    img["id"]: img["file_name"]
    for img in data["images"]
}

In [10]:
image_paths = []
captions = []

for ann in data["annotations"]:

    img_file = image_id_to_file[ann["image_id"]]

    image_paths.append(os.path.join(train_img_dir, img_file))
    captions.append(ann["caption"])

In [11]:
train_imgs, val_imgs, train_caps, val_caps = train_test_split(
    image_paths,
    captions,
    test_size=0.2,
    random_state=42
)

In [12]:
class Vocabulary:

    def __init__(self, freq_threshold):

        self.itos = {
            0: "<PAD>",
            1: "<SOS>",
            2: "<EOS>",
            3: "<UNK>"
        }

        self.stoi = {
            "<PAD>": 0,
            "<SOS>": 1,
            "<EOS>": 2,
            "<UNK>": 3
        }

        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def tokenizer(self, text):
        return text.lower().split()

    def build_vocabulary(self, sentence_list):

        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:

            for word in self.tokenizer(sentence):

                frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:

                    self.stoi[word] = idx
                    self.itos[idx] = word

                    idx += 1

In [13]:
vocab = Vocabulary(freq_threshold=2)

vocab.build_vocabulary(train_caps)

In [14]:
class CocoDataset(Dataset):

    def __init__(self, image_paths, captions, vocab, transform=None):

        self.image_paths = image_paths
        self.captions = captions
        self.vocab = vocab
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")
        caption = self.captions[idx]

        if self.transform:
            image = self.transform(image)

        # tokenization → indices
        numericalized = [self.vocab.stoi["<SOS>"]]

        numericalized += [
            self.vocab.stoi.get(word, self.vocab.stoi["<UNK>"])
            for word in caption.split()
        ]

        numericalized.append(self.vocab.stoi["<EOS>"])

        caption_tensor = torch.tensor(numericalized)

        return image, caption_tensor

In [15]:
def get_transform():

    return transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.RandomCrop((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.5, 0.5, 0.5),
            (0.5, 0.5, 0.5)
        )
    ])
    

In [16]:
train_dataset = CocoDataset(
    train_imgs,
    train_caps,
    vocab,
    transform=get_transform()
)

val_dataset = CocoDataset(
    val_imgs,
    val_caps,
    vocab,
    transform=get_transform()
)

Setting up the models

In [17]:
# /root/.cache/kagglehub/datasets/nikhil7280/coco-image-caption/versions/1      <--- path to dataset incase we forget


class EncodeCNN(nn.Module):

    def __init__(self, embed_size, train_CNN=False):

        super(EncodeCNN, self).__init__()

        self.train_CNN = train_CNN

        self.inception = models.inception_v3(
            weights="DEFAULT",
            aux_logits=True
        )

        # disable auxiliary classifier
        self.inception.aux_logits = False
        self.inception.AuxLogits = None

        # replace final layer
        self.inception.fc = nn.Linear(
            self.inception.fc.in_features,
            embed_size
        )

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):

        features = self.inception(images)

        for name, param in self.inception.named_parameters():

            if "fc" in name:
                param.requires_grad = True
            else:
                param.requires_grad = self.train_CNN

        return self.dropout(self.relu(features))



class DecodeRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(DecodeRNN, self).__init__()

        self.embed = nn.Embedding(vocab_size, embed_size)  
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)
    #New forward function
    def forward(self, features, captions):
        embeddings = self.dropout(self.embed(captions))
        embeddings = embeddings.permute(1, 0, 2)
    
        embeddings = torch.cat(
            (features.unsqueeze(0), embeddings),
            dim=0
        )
    
        outputs, _ = self.lstm(embeddings)
        outputs = self.linear(outputs)
    
        # Return only the caption timesteps, not the CNN feature timestep
        return outputs[1:]
        # embeddings now (seq_len+1, batch, embed)
        outputs, _ = self.lstm(embeddings)
        outputs = self.linear(outputs)
        return outputs
'''
    def forward(self, features, captions):

        embeddings = self.dropout(self.embed(captions))

        embeddings = torch.cat(
            (features.unsqueeze(0), embeddings),
            dim=0
        )

        hiddens, _ = self.lstm(embeddings)

        output = self.linear(hiddens)

        return output  '''
    

class CNNtoRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(CNNtoRNN, self).__init__()

        self.encoderCNN = EncodeCNN(embed_size)
        self.decoderRNN = DecodeRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):

        features = self.encoderCNN(images)
        outputs = self.decoderRNN(features, captions)

        return outputs

    def caption_image(self, image, vocab, max_length=50):

        res_caption = []

        with torch.no_grad():

            x = self.encoderCNN(image).unsqueeze(0)
            states = None

            for _ in range(max_length):

                hiddens, states = self.decoderRNN.lstm(x, states)

                output = self.decoderRNN.linear(hiddens.squeeze(0))

                predicted = output.argmax(1)

                res_caption.append(predicted.item())

                x = self.decoderRNN.embed(predicted).unsqueeze(0)

                if vocab.itos[predicted.item()] == "<EOS>":
                    break

        return [vocab.itos[idx] for idx in res_caption]

In [18]:
def collate_fn(batch):
    images = []
    captions = []
    for img, cap in batch:
        images.append(img)
        captions.append(cap.detach().clone() if isinstance(cap, torch.Tensor) else torch.as_tensor(cap))

    images   = torch.stack(images, dim=0)
    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=0
    )
    return images, captions

In [19]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn   
)

In [20]:
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,      # no need to shuffle for evaluation
    num_workers=2,
    collate_fn=collate_fn
)

Training code

In [21]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for imgs, captions in loader:

            imgs = imgs.to(device)
            captions = captions.to(device)

            # input / target (teacher forcing)
            inputs = captions[:, :-1]
            targets = captions[:, 1:]
            
            outputs = model(imgs, inputs)

            loss = criterion(
                outputs.reshape(-1, outputs.shape[2]),
                targets.reshape(-1)
            )
            total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"  Val Loss : {avg_loss:.4f}")

    model.train()
    return avg_loss

In [22]:
def train():

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    embed_size = 256
    hidden_size = 256
    vocab_size = len(train_dataset.vocab)
    num_layers = 1
    lr = 3e-4
    num_epochs = 10
    best_loss = float("inf")
    logging = True
    model = CNNtoRNN(
        embed_size,
        hidden_size,
        vocab_size,
        num_layers
    ).to(device)

    criterion = nn.CrossEntropyLoss(
        ignore_index=train_dataset.vocab.stoi["<PAD>"]
    )

    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(num_epochs):

        total_loss = 0

        for idx, (imgs, captions) in enumerate(train_loader):

            imgs = imgs.to(device)
            captions = captions.to(device)

            # input / target (teacher forcing)
            inputs = captions[:, :-1]
            targets = captions[:, 1:]

            outputs = model(imgs, inputs)

            loss = criterion(
                outputs.reshape(-1, outputs.shape[2]),
                targets.reshape(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if idx % 100 == 0:
                print(
                    f"Epoch [{epoch+1}/{num_epochs}] "
                    f"Step [{idx}] "
                    f"Loss: {loss.item():.4f}"
                )
            if idx % 1000 == 0:
                eval_loss = evaluate(model, val_loader, criterion, device)
                if eval_loss < best_loss:
                    best_loss = eval_loss
                    torch.save({                            #Lets us load the model later like this:checkpoint = torch.load("best_model.pth", map_location=device), model = CNNtoRNN(embed_size, hidden_size, vocab_size, num_layers).to(device),model.load_state_dict(checkpoint["model"]), optimizer.load_state_dict(checkpoint["optimizer"]) 
                        "epoch":      epoch + 1,
                        "step":       idx,
                        "model":      model.state_dict(),
                        "optimizer":  optimizer.state_dict(),
                        "val_loss":   eval_loss,
                        "vocab":      vocab,
                    }, "/Lab3/Models/BestModel.pth")
                    print(f"Saved best model (val_loss={eval_loss:.4f})")
                if logging:
                    wandb.log({"Train Loss": loss.item(), "Val Loss": eval_loss, "epoch": epoch+1, "step": idx})
        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

    return model

In [ ]:
import wandb
wandb.init(project="Lab3", name="Test_run2", config={
    "learning_rate": 3e-4,
    "batch_size": 32,
    "epochs": 10,
})
model = train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch [1/10] Step [0] Loss: 9.8256


In [ ]:
model.eval()

In [ ]:
for imgs, caps in train_loader:
    print(imgs.shape)
    print(caps.shape)
    break

In [ ]:
print(f"Steps per epoch: {len(train_loader)}")
print(f"Steps per epoch: {len(val_loader)}")
wandb.finish()

In [ ]:
import os
os.makedirs("/Lab3/Models", exist_ok=True)